In [4]:
import pandas as pd
import numpy as np
from itertools import combinations

In [5]:
# Load CSVs into DataFrames
users_df = pd.read_csv("users.csv")
groups_df = pd.read_csv("groups.csv")
memberships_df = pd.read_csv("group_memberships.csv")

In [7]:



# ---------------------------
# Helpers
# ---------------------------
def parse_age_range(s):
    """'19-24' -> (19, 24)"""
    if pd.isna(s):
        return (None, None)
    a, b = str(s).split("-")
    return (int(a), int(b))

def jaccard_array(a, b):
    """Jaccard for arrays like ['English','French']"""
    if a is None or b is None:
        return 0.0
    A, B = set(a), set(b)
    if not A and not B:
        return 0.0
    return len(A & B) / len(A | B)

def lifestyle_similarity(a, b):
    """
    Similarity for levels 1..10: average of smoking/drinking/weed.
    Similarity = 1 - |diff|/9, so equal -> 1.0, far apart -> ~0
    """
    sims = []
    for k in ("smoking_level", "drinking_level", "weed_level"):
        av, bv = a[k], b[k]
        if pd.isna(av) or pd.isna(bv):
            sims.append(0.0)
        else:
            sims.append(1 - abs(int(av) - int(bv)) / 9.0)
    return float(np.mean(sims))

def group_size(row):
    return int(row["num_men"] + row["num_women"] + row["num_nonbinary"])

def size_compat(a, b):
    """
    Penalize deviation from each other's ideal group size.
    Normalize by 10 so score stays in [0,1] for typical sizes.
    """
    a_ideal = int(a.get("ideal_group_size", 0) or 0)
    b_ideal = int(b.get("ideal_group_size", 0) or 0)
    a_size = group_size(a)
    b_size = group_size(b)
    # If ideals are zero/missing, treat as neutral (no penalty)
    if a_ideal == 0 and b_ideal == 0:
        return 1.0
    penalty = 0.0
    if a_ideal:
        penalty += abs(a_ideal - b_size)
    if b_ideal:
        penalty += abs(b_ideal - a_size)
    penalty /= 2.0
    return max(0.0, min(1.0, 1.0 - (penalty / 10.0)))

def age_overlap_score(a, b):
    """Jaccard-like overlap for numeric ranges."""
    a_min, a_max = parse_age_range(a["age_range"])
    b_min, b_max = parse_age_range(b["age_range"])
    if None in (a_min, a_max, b_min, b_max):
        return 0.0
    overlap = max(0, min(a_max, b_max) - max(a_min, b_min))
    union = max(a_max, b_max) - min(a_min, b_min)
    return (overlap / union) if union > 0 else 0.0

def inclusivity_score(a, b):
    """Both true -> 1.0; one true -> 0.5; none -> 0.0."""
    sa = bool(a.get("sexuality_inclusive", False))
    sb = bool(b.get("sexuality_inclusive", False))
    if sa and sb:
        return 1.0
    if sa or sb:
        return 0.5
    return 0.0

def accessibility_score(a, b):
    """Both true -> 1.0; one true -> 0.5; none -> 0.0."""
    aa = bool(a.get("accessibility_friendly", False))
    ab = bool(b.get("accessibility_friendly", False))
    if aa and ab:
        return 1.0
    if aa or ab:
        return 0.5
    return 0.0

def rating_score(a, b):
    """Average of ratings normalized to [0,1] given ratings out of 5."""
    ra = float(a.get("group_rating") or 0.0)
    rb = float(b.get("group_rating") or 0.0)
    return (ra + rb) / 10.0

def city_component(a, b, same_city_only=False):
    """Same city bonus; can be a hard gate if same_city_only=True."""
    same = str(a.get("location")) == str(b.get("location"))
    if same_city_only:
        return 1.0 if same else 0.0
    return 1.0 if same else 0.3

# ---------------------------
# Main matching function
# ---------------------------
DEFAULT_WEIGHTS = {
    "size": 0.20,
    "age": 0.15,
    "lifestyle": 0.20,
    "languages": 0.15,
    "inclusivity_access": 0.10,
    "rating": 0.10,
    "location": 0.10,
}

def compute_pair_scores(groups_df: pd.DataFrame,
                        weights: dict = None,
                        same_city_only: bool = False) -> pd.DataFrame:
    """
    Returns a DataFrame with all unordered pairs and a 0–100 match_score,
    plus the sub-scores for explainability.
    """
    w = (weights or DEFAULT_WEIGHTS).copy()
    # Ensure required columns exist
    required = [
        "id","age_range","num_men","num_women","num_nonbinary","location",
        "smoking_level","drinking_level","weed_level","ideal_group_size",
        "languages","sexuality_inclusive","accessibility_friendly","group_rating"
    ]
    missing = [c for c in required if c not in groups_df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    # Convert languages to Python lists if needed
    def to_list(v):
        if isinstance(v, list):
            return v
        if pd.isna(v):
            return []
        # Accept Postgres-style: {"English","French"} or {English,French}
        s = str(v).strip()
        s = s.strip("{}")
        if not s:
            return []
        return [x.strip().strip('"') for x in s.split(",")]

    rows = []
    # Create an index for quick row lookup by id
    by_id = {int(r["id"]): r for _, r in groups_df.assign(
        languages=groups_df["languages"].apply(to_list)
    ).iterrows()}

    for a_id, b_id in combinations(sorted(by_id.keys()), 2):
        a, b = by_id[a_id], by_id[b_id]

        size_score = size_compat(a, b)
        age_score = age_overlap_score(a, b)
        life_score = lifestyle_similarity(a, b)
        lang_score = jaccard_array(a.get("languages", []), b.get("languages", []))
        inc = inclusivity_score(a, b)
        acc = accessibility_score(a, b)
        inc_acc = (inc + acc) / 2.0
        rate = rating_score(a, b)
        city = city_component(a, b, same_city_only=same_city_only)

        match_score = (
            w["size"] * size_score +
            w["age"] * age_score +
            w["lifestyle"] * life_score +
            w["languages"] * lang_score +
            w["inclusivity_access"] * inc_acc +
            w["rating"] * rate +
            w["location"] * city
        ) * 100.0

        rows.append({
            "a_id": a_id, "b_id": b_id,
            "match_score": round(match_score, 1),
            "size_score": round(size_score, 3),
            "age_score": round(age_score, 3),
            "lifestyle_score": round(life_score, 3),
            "lang_score": round(lang_score, 3),
            "inclusivity_access_score": round(inc_acc, 3),
            "rating_score": round(rate, 3),
            "city_score": round(city, 3),
        })

    return pd.DataFrame(rows).sort_values(["match_score","a_id","b_id"], ascending=[False, True, True]).reset_index(drop=True)

def best_match_per_group(pair_scores: pd.DataFrame) -> pd.DataFrame:
    """Pick the top-scoring partner for each group."""
    # make asymmetric view (A->B and B->A)
    a_view = pair_scores.rename(columns={"a_id": "group_id", "b_id": "other_id"})
    b_view = pair_scores.rename(columns={"b_id": "group_id", "a_id": "other_id"})
    both = pd.concat([
        a_view[["group_id","other_id","match_score"]],
        b_view[["group_id","other_id","match_score"]],
    ], ignore_index=True)
    # rank by match_score desc
    both["rnk"] = both.groupby("group_id")["match_score"].rank(method="min", ascending=False)
    return both[both["rnk"] == 1].sort_values("group_id")[["group_id","other_id","match_score"]].reset_index(drop=True)

# ---------------------------
# Example usage
# ---------------------------


pair_scores = compute_pair_scores(groups_df, same_city_only=False)
print("All pair scores:")
print(pair_scores)

best = best_match_per_group(pair_scores)
print("\nBest match per group:")
print(best)


All pair scores:
   a_id  b_id  match_score  size_score  age_score  lifestyle_score  \
0     2     4         84.7         0.7      0.714            0.852   
1     1     3         76.4         0.8      0.667            0.852   
2     1     2         70.1         0.9      0.375            0.889   
3     2     3         69.9         0.9      0.571            0.741   
4     3     4         62.1         0.8      0.375            0.593   
5     1     4         58.9         0.6      0.222            0.741   

   lang_score  inclusivity_access_score  rating_score  city_score  
0       1.000                       1.0          0.79         1.0  
1       0.333                       1.0          0.84         1.0  
2       0.500                       1.0          0.82         0.3  
3       0.500                       1.0          0.80         0.3  
4       0.500                       1.0          0.81         0.3  
5       0.500                       1.0          0.83         0.3  

Best match per 